# Build the pixel cache

Decodes the competition DICOM into a `uint8` cache small enough to train on locally. This is
the only step needing the full ~930 GB, which is why it runs here.

Attach the competition data and the `rsnaknee-src` dataset. Accelerator **None** — the job is
decode-bound, so a GPU idles and burns quota.

In [ ]:
import sys, subprocess, shutil
from pathlib import Path

# Discover both roots by the files that identify them, rather than by assuming a mount layout.
# Kaggle nests these under /kaggle/input/datasets/<user>/<slug> and /kaggle/input/competitions/
# <slug>, but that has changed before and a wrong guess is indistinguishable from "not attached".
INPUT = Path('/kaggle/input')

PKG = next((m.parent.parent for m in INPUT.glob('**/rsnaknee/__init__.py')), None)
DATA = next((m.parent for m in INPUT.glob('**/train_series.csv')), None)

if PKG is None or DATA is None:
    print('PKG =', PKG, '| DATA =', DATA)
    print('mounts under /kaggle/input:')
    for d in sorted(INPUT.rglob('*')):
        if d.is_dir() and len(d.relative_to(INPUT).parts) <= 3:
            print('  ', d)
    raise SystemExit('Attach both the competition and rsnaknee-src, then re-run.')

shutil.copytree(PKG, '/kaggle/working/pkg', dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working/pkg')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg', 'python-gdcm'], check=False)

import rsnaknee, pydicom
print('package :', PKG)
print('data    :', DATA)
print('rsnaknee', rsnaknee.__version__, '| pydicom', pydicom.__version__)
print(sorted(p.name for p in DATA.iterdir())[:8])

In [ ]:
IMAGE_SIZE = 224   # divisible by 14 for DINOv2; cache grows with the square of this
SLICES     = 8     # centred band per slot; linear in cache size, the cheap axis
WORKERS    = 4
OUT        = Path('/kaggle/working/cache/r224s8')

import numpy as np, pandas as pd, time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

from rsnaknee.cache import CacheMeta, write_meta
from rsnaknee.dicom_io import load_series, read_header, study_side
from rsnaknee.model import SLOTS

def slot_of(plane, fluid):
    for i, (p, f) in enumerate(SLOTS):
        if p == plane and f == int(fluid):
            return i
    return None

def build_one(job):
    study, rows, n_slices, image_size = job
    out  = np.zeros((len(SLOTS), n_slices, image_size, image_size), dtype=np.uint8)
    mask = np.zeros(len(SLOTS), dtype=bool)
    by_slot, headers = defaultdict(list), []
    for row in rows:
        slot = slot_of(row['Anatomical_Plane'], row['Fluid_Sensitive'])
        if slot is None:
            continue
        paths = sorted((DATA / 'train_series' / study / row['SeriesInstanceUID']).glob('*.dcm'))
        if not paths:
            continue
        by_slot[slot].append((row['Anatomical_Plane'], paths))
        headers.extend(h for h in (read_header(p) for p in paths[::8]) if h is not None)
    # Laterality is decided once per study: the knee does not change between sequences, and a
    # per-series vote lets one noisy stack mirror half a study.
    side = study_side(headers)
    for slot, cands in by_slot.items():
        plane, paths = max(cands, key=lambda c: len(c[1]))
        stack = load_series(paths, plane=plane, side=side, n_slices=n_slices, image_size=image_size)
        if stack is not None:
            out[slot], mask[slot] = stack, True
    return study, out, mask

series  = pd.read_csv(DATA / 'train_series.csv')
studies = sorted(series['StudyInstanceUID'].unique())
grouped = {u: r.to_dict('records') for u, r in series.groupby('StudyInstanceUID')}
print(len(studies), 'studies,', len(series), 'series')

In [ ]:
# Preflight on three studies. A full decode is hours; discovering a path or geometry bug at the
# end of one is the expensive way to learn it. This costs seconds and fails loudly.
t0 = time.time()
probe = [build_one((u, grouped[u], SLICES, IMAGE_SIZE)) for u in studies[:3]]
per_study = (time.time() - t0) / 3

for uid, stack, mask in probe:
    print(f'{uid[:20]}... slots={mask.sum()}/{len(SLOTS)} nonzero={(stack > 0).mean():.2f}')

assert any(m.sum() for _, _, m in probe), 'No slots decoded — check slot assignment.'
assert any((s > 0).mean() > 0.1 for _, s, _ in probe), 'Decoded images are essentially blank.'
print(f'\n{per_study:.1f}s/study serial -> ~{per_study * len(studies) / WORKERS / 60:.0f} min on {WORKERS} workers')

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
from rsnaknee.cache import SHARD_STUDIES

# Write pixels in ~1 GB shards. Kaggle's output endpoint resets the connection partway through
# a single ~10 GB file ("Stream 1 was reset by remote peer"), which loses the whole decode; a
# failed shard is retryable on its own.
masks = np.zeros((len(studies), len(SLOTS)), dtype=bool)
order = {u: i for i, u in enumerate(studies)}
bounds = list(range(0, len(studies), SHARD_STUDIES)) + [len(studies)]
shards = [
    np.lib.format.open_memmap(
        OUT / f'pixels_{k:03d}.npy', mode='w+', dtype=np.uint8,
        shape=(bounds[k + 1] - bounds[k], len(SLOTS), SLICES, IMAGE_SIZE, IMAGE_SIZE))
    for k in range(len(bounds) - 1)
]
print(f'{len(shards)} shards of <= {SHARD_STUDIES} studies '
      f'({shards[0].nbytes / 1024**3:.2f} GB each)')

def place(i, stack):
    k = min(i // SHARD_STUDIES, len(shards) - 1)
    shards[k][i - bounds[k]] = stack

start = time.time()
jobs = [(u, grouped[u], SLICES, IMAGE_SIZE) for u in studies]
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    for uid, stack, mask in tqdm(pool.map(build_one, jobs), total=len(jobs)):
        i = order[uid]; place(i, stack); masks[i] = mask
for s in shards:
    s.flush()

np.save(OUT / 'mask.npy', masks)
pd.Series(studies, name='StudyInstanceUID').to_frame().to_parquet(OUT / 'index.parquet')
write_meta(OUT, CacheMeta(image_size=IMAGE_SIZE, slices=SLICES, n_slots=len(SLOTS),
    n_studies=len(studies), laterality_normalised=True, slice_rule=f'centred-{SLICES}'))

total = sum(s.nbytes for s in shards)
print(f'{(time.time() - start) / 60:.1f} min | {total / 1024**3:.1f} GB in {len(shards)} shards')
print('mean slots present  :', masks.sum(axis=1).mean().round(2), 'of', len(SLOTS))
print('studies with 0 slots:', int((~masks.any(axis=1)).sum()))

## Check before downloading

`mean slots present` should be 4–5 of 6. `studies with 0 slots` must be near zero — those train
on all-zero images that look like valid data.

In [ ]:
import matplotlib.pyplot as plt

i = int(np.argmax(masks[:len(shards[0])].sum(axis=1)))  # within shard 0
present = np.flatnonzero(masks[i])
fig, axes = plt.subplots(1, len(present), figsize=(3 * len(present), 3.4))
for ax, slot in zip(np.atleast_1d(axes), present):
    ax.imshow(shards[0][i, slot, SLICES // 2], cmap='gray')
    ax.set_title(f'{SLOTS[slot][0]}  fluid={SLOTS[slot][1]}', fontsize=9)
    ax.axis('off')
plt.suptitle('laterality-normalised slots, one study')
plt.tight_layout(); plt.show()